In order to understand the mechanisms behind the recommender system, we consider a toy example.

In [15]:
import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD
import matplotlib.pyplot as plt

users = ["u1", "u2", "u3", "u4", "u5"]
items = ["Politics", "Tech", "Sports", "Finance", "Health", "Travel"]

R = np.array([
    [1, 1, 0, 0, 0, 0],
    [1, 1, 0, 1, 0, 0],
    [0, 0, 1, 0, 1, 0],
    [0, 0, 1, 0, 1, 1],
    [1, 0, 0, 1, 0, 0],
])

df = pd.DataFrame(R, index=users, columns=items)
df

,Politics,Tech,Sports,Finance,Health,Travel
u1,1,1,0,0,0,0
u2,1,1,0,1,0,0
u3,0,0,1,0,1,0
u4,0,0,1,0,1,1
u5,1,0,0,1,0,0


In [17]:
k = 2

svd = TruncatedSVD(n_components=k, random_state=42)
P = svd.fit_transform(R)
Q = svd.components_.T

S_hat = P @ Q.T
score_df = pd.DataFrame(S_hat, index=users, columns=items)
print(score_df.round(2))

def show_factors(P, Q, users, items, k):
    latent_cols = [f"z{i+1}" for i in range(k)]
    P_df = pd.DataFrame(P, index=users, columns=latent_cols).round(2)
    Q_df = pd.DataFrame(Q, index=items, columns=latent_cols).round(2)
    return P_df, Q_df

P_df, Q_df = show_factors(P, Q, users, items, k)
print("User latent factors (P):")
print(P_df)
print("Item latent factors (Q):")
print(Q_df)

    Politics  Tech  Sports  Finance  Health  Travel
u1      0.85  0.60    0.00     0.60    0.00    0.00
u2      1.21  0.85    0.00     0.85    0.00    0.00
u3     -0.00  0.00    0.86    -0.00    0.86    0.49
u4     -0.00  0.00    1.11    -0.00    1.11    0.62
u5      0.85  0.60   -0.00     0.60   -0.00   -0.00
User latent factors (P):
      z1    z2
u1  1.21  0.00
u2  1.71 -0.00
u3  0.00  1.31
u4  0.00  1.68
u5  1.21 -0.00
Item latent factors (Q):
            z1    z2
Politics  0.71 -0.00
Tech      0.50  0.00
Sports    0.00  0.66
Finance   0.50 -0.00
Health    0.00  0.66
Travel    0.00  0.37


With k=2, we see that the model is able to effectively create two unique profiles: one for users u1, u2, and u5, and another for users u3 and u4. Notably, users u1 and u5 ended up with the same recommendation array. This is due to the fact that they ahd the same level of overlap with user u2, both having two items in common. User u2 acts as a connector for u1 and u5 to be recommended the one they haven't read, but we can also see the impact of users u1 and u5 in the lower values for Tech and Finance compared to politics.

This is a relatively clean example since it is fairly easy to group users into two distinct groups with k=2.

In [18]:
R2 = np.array([
    [1, 1, 0, 0, 0, 0],
    [1, 1, 1, 1, 0, 0],
    [0, 0, 1, 0, 1, 0],
    [0, 0, 1, 0, 1, 1],
    [1, 0, 0, 1, 0, 0],
])

df2 = pd.DataFrame(R2, index=users, columns=items)

P2 = svd.fit_transform(R2)
Q2 = svd.components_.T

S_hat2 = P2 @ Q2.T
score_df2 = pd.DataFrame(S_hat2, index=users, columns=items)
print(score_df2.round(2))

P2_df, Q2_df = show_factors(P2, Q2, users, items, k)
print("User latent factors (P2):")
print(P2_df)
print("Item latent factors (Q2):")
print(Q2_df)

    Politics  Tech  Sports  Finance  Health  Travel
u1      0.81  0.57    0.18     0.57   -0.16   -0.10
u2      1.24  0.90    0.78     0.90    0.21    0.11
u3     -0.02  0.04    0.92     0.04    0.82    0.46
u4     -0.09  0.01    1.16     0.01    1.05    0.59
u5      0.81  0.57    0.18     0.57   -0.16   -0.10
User latent factors (P2):
      z1    z2
u1  0.98 -0.65
u2  1.91 -0.41
u3  0.75  1.09
u4  0.87  1.43
u5  0.98 -0.65
Item latent factors (Q2):
            z1    z2
Politics  0.56 -0.40
Tech      0.42 -0.25
Sports    0.51  0.50
Finance   0.42 -0.25
Health    0.23  0.59
Travel    0.13  0.34


Here we can see the impact of simply adding in that user u2 has read the Sports article. The previously distinct groups now have overlap, and the model has to adjust to account for that. There are still pretty clear distinctions between the groups of users u1, u2, and u5 and users u3 and u4, but there has been some uncertainty added. It is easy to understand how the scores for each user-item pair makes sense, but the P2 and Q2 matrices would take more consideration to understand compared to the original P and Q matrices.

In [11]:
def recommend_for_user(user_id, R, scores, users, items, top_k=3):
    u_idx = users.index(user_id)
    seen = R[u_idx] > 0

    user_scores = scores[u_idx].copy()
    user_scores[seen] = -np.inf

    top_indices = np.argsort(user_scores)[::-1][:top_k]
    return [(items[i], round(float(user_scores[i]),2)) for i in top_indices]

print(recommend_for_user("u1", R, S_hat, users, items, top_k=3))
print(recommend_for_user("u1", R2, S_hat2, users, items, top_k=3))

[('Finance', 0.6), ('Health', 0.0), ('Sports', 0.0)]
[('Finance', 0.57), ('Sports', 0.18), ('Travel', -0.1)]


We can see that, for the first model, there is a very clear recommendation for Finance and no other meaningful recommendations. For the second model, the recommendation for Finance is only slightly lower, while a new weak recommendation for Sports exists. The second model also has a negative score for Travel. Negative scores should not be interpreted as predicted dislike for a news item, simply as a lower recommendation score.